# 5-3. 현대적 레시피 — 2017년과 지금의 차이

> **이 노트북에서 답할 질문**
> 원논문 트랜스포머와 Llama 같은 최신 모델은 무엇이 다르고, 왜 바꿨는가?

## 한눈에 보는 변화

| 부품 | 원논문 (2017) | 현대 (Llama 계열) | 바꾼 이유 |
|---|---|---|---|
| 정규화 위치 | Post-LN | **Pre-LN** | 학습 안정성 |
| 정규화 방식 | LayerNorm | **RMSNorm** | 더 빠름 |
| 위치 정보 | 사인/코사인 절대 | **RoPE (상대·회전)** | 길이 확장성 |
| FFN 활성화 | ReLU | **SwiGLU** | 성능 |
| 어텐션 KV | MHA (헤드마다) | **GQA (공유)** | 추론 메모리 |
| 편향(bias) | 있음 | **대부분 제거** | 불필요 |

하나씩 **왜**를 확인합니다. Pre-LN은 5-2에서 다뤘으니 나머지를 봅니다.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "common.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import math, time
import torch
import torch.nn as nn
import torch.nn.functional as F
from common import set_seed, get_device, count_params, human

set_seed(1234)
device = get_device()

## 1. RMSNorm — 평균 빼기를 생략하다

LayerNorm은 두 가지를 합니다: **평균 빼기**(centering)와 **분산 나누기**(scaling).

$$\text{LayerNorm}(x) = \gamma \cdot \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta$$

RMSNorm은 **평균 빼기를 생략**합니다. 실험적으로 성능 차이가 거의 없는데 더 빠릅니다.

$$\text{RMSNorm}(x) = \gamma \cdot \frac{x}{\sqrt{\frac{1}{d}\sum x_i^2 + \epsilon}}$$

평균 계산, 뺄셈, 그리고 $\beta$ 파라미터가 사라집니다.

In [ ]:
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))     # gamma 만 있고 beta 는 없음

    def forward(self, x):
        rms = x.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return self.weight * (x * rms)

dim = 4096
ln, rms = nn.LayerNorm(dim), RMSNorm(dim)
print(f"LayerNorm 파라미터: {human(count_params(ln))}  (gamma + beta)")
print(f"RMSNorm   파라미터: {human(count_params(rms))}  (gamma 만)")

x = torch.randn(64, 512, dim)
for name, mod in (("LayerNorm", ln), ("RMSNorm", rms)):
    t0 = time.perf_counter()
    for _ in range(10):
        mod(x)
    print(f"{name:>10}: {(time.perf_counter()-t0)/10*1000:6.1f} ms / 회 (CPU)")

> 이 정도 차이가 대수롭지 않아 보일 수 있습니다. 하지만 **층마다 2번씩, 수십 층**,
> 그리고 **수조 개 토큰**을 학습시킨다고 생각해 보세요. 누적되면 큰 비용입니다.

## 2. RoPE — 위치를 "회전"으로 넣기

5-2에서 본 학습된 절대 위치 임베딩의 문제:

- `block_size` 를 넘으면 임베딩이 **아예 없음** → 문맥 확장 불가
- **절대 위치**만 알고 상대 거리는 모름. "3칸 앞"이라는 개념이 없음

### RoPE의 아이디어

임베딩에 무언가를 **더하는** 대신, Query와 Key 벡터를 **위치만큼 회전**시킵니다.

2차원 쌍마다 각도 $m\theta$ 만큼 돌립니다($m$ = 위치):

$$\begin{pmatrix} q'_1 \\ q'_2 \end{pmatrix} =
\begin{pmatrix} \cos m\theta & -\sin m\theta \\ \sin m\theta & \cos m\theta \end{pmatrix}
\begin{pmatrix} q_1 \\ q_2 \end{pmatrix}$$

### 왜 이게 우아한가

회전된 두 벡터의 **내적이 두 위치의 차이에만 의존**합니다.

$$\langle R_m q, R_n k \rangle = f(q, k, m - n)$$

어텐션 점수가 곧 $QK^\top$ 이니, **어텐션이 자동으로 상대 거리를 보게 됩니다.**
더한 게 아니라 곱한(회전시킨) 덕분입니다.

In [ ]:
def build_rope_cache(seq_len, head_dim, base=10000.0):
    """위치별 cos/sin 값을 미리 계산해 둡니다."""
    # 차원 쌍마다 다른 주파수: 낮은 차원은 빠르게, 높은 차원은 느리게 회전
    theta = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    pos = torch.arange(seq_len).float()
    freqs = torch.outer(pos, theta)            # (seq_len, head_dim/2)
    return freqs.cos(), freqs.sin()


def apply_rope(x, cos, sin):
    """x: (B, T, H, head_dim) 를 위치만큼 회전시킵니다."""
    T = x.shape[1]
    x1, x2 = x[..., 0::2], x[..., 1::2]        # 짝수/홀수 차원을 2D 쌍으로
    c = cos[:T].view(1, T, 1, -1)
    s = sin[:T].view(1, T, 1, -1)
    out = torch.stack([x1 * c - x2 * s,        # 회전 행렬 적용
                       x1 * s + x2 * c], dim=-1)
    return out.flatten(-2)

head_dim, T = 8, 16
cos, sin = build_rope_cache(T, head_dim)
q = torch.randn(1, T, 1, head_dim)
q_rot = apply_rope(q, cos, sin)
print(f"회전 전 {tuple(q.shape)} -> 회전 후 {tuple(q_rot.shape)}  (모양 그대로)")
print(f"벡터 길이가 보존되는가? {torch.allclose(q.norm(dim=-1), q_rot.norm(dim=-1), atol=1e-5)}")
print("  (회전은 길이를 바꾸지 않습니다 -- 정보를 잃지 않는다는 뜻)")

In [ ]:
# 핵심 성질 확인: 내적이 '위치 차이'에만 의존하는가?
set_seed(0)
qv = torch.randn(1, 1, 1, head_dim)
kv = torch.randn(1, 1, 1, head_dim)

def dot_at(m, n):
    cos_, sin_ = build_rope_cache(max(m, n) + 1, head_dim)
    qm = apply_rope(qv.expand(1, m + 1, 1, head_dim).clone(), cos_, sin_)[0, m]
    kn = apply_rope(kv.expand(1, n + 1, 1, head_dim).clone(), cos_, sin_)[0, n]
    return (qm * kn).sum().item()

print(f"{'(m, n)':>12} | {'m - n':>6} | {'내적':>10}")
print("-" * 34)
for m, n in [(5, 3), (10, 8), (50, 48), (7, 2), (20, 15)]:
    print(f"{str((m, n)):>12} | {m-n:>6} | {dot_at(m, n):>10.4f}")
print(f"{chr(10)}m-n 이 같으면 내적이 같습니다. 절대 위치가 아무리 멀어도요.")
print("-> 학습 때 본 적 없는 위치에서도 '상대 거리' 감각이 유지됩니다.")

### 이것이 긴 문맥의 비결입니다

학습을 4096 토큰으로 했어도, RoPE는 **주파수를 조정**하는 것만으로
32K, 128K로 확장할 수 있습니다(`base` 값을 키우는 방식 등).

절대 위치 임베딩으로는 불가능한 일입니다. 없는 임베딩을 만들어 낼 수 없으니까요.

> 요즘 모델 카드에서 보는 `rope_theta: 500000` 같은 설정이 바로 이 `base` 값입니다.

## 3. SwiGLU — 게이트가 달린 FFN

5-2의 FFN은 이랬습니다.

$$\text{FFN}(x) = W_2\,\text{ReLU}(W_1 x)$$

SwiGLU는 **두 갈래로 나눠서, 한쪽이 다른 쪽의 문을 여닫게** 합니다.

$$\text{SwiGLU}(x) = W_2\,\big(\text{SiLU}(W_1 x) \odot W_3 x\big)$$

- $\text{SiLU}(z) = z \cdot \sigma(z)$ — ReLU와 달리 음수 구간에서도 매끄럽게 흐름
- $\odot$ — 원소별 곱. **게이트**: 어떤 정보를 통과시킬지 학습으로 조절

In [ ]:
import matplotlib.pyplot as plt

z = torch.linspace(-5, 5, 200)
plt.figure(figsize=(7, 3.5))
plt.plot(z, F.relu(z), label="ReLU")
plt.plot(z, F.silu(z), label="SiLU (Swish)")
plt.plot(z, F.gelu(z), label="GELU", ls="--", alpha=0.7)
plt.axhline(0, c="gray", lw=0.5); plt.axvline(0, c="gray", lw=0.5)
plt.legend(); plt.grid(alpha=0.3); plt.title("활성화 함수 비교")
plt.tight_layout(); plt.show()

print("ReLU는 음수를 전부 0으로 죽입니다 -> 그 뉴런에 기울기가 흐르지 않습니다.")
print("SiLU/GELU는 음수 구간에서도 조금씩 흘려보냅니다.")

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, dim, hidden=None):
        super().__init__()
        # 가중치 행렬이 3개가 되므로, 파라미터를 맞추려 hidden 을 2/3 로 줄입니다
        hidden = hidden or int(2 / 3 * 4 * dim)
        hidden = 64 * ((hidden + 63) // 64)          # 하드웨어 친화적으로 64의 배수
        self.w1 = nn.Linear(dim, hidden, bias=False)   # 게이트 쪽
        self.w3 = nn.Linear(dim, hidden, bias=False)   # 값 쪽
        self.w2 = nn.Linear(hidden, dim, bias=False)   # 되돌리기

    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

class ClassicFFN(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, 4 * dim), nn.ReLU(), nn.Linear(4 * dim, dim))
    def forward(self, x):
        return self.net(x)

dim = 512
a, b = ClassicFFN(dim), SwiGLU(dim)
print(f"기존 FFN (ReLU, 4x) : {human(count_params(a))}")
print(f"SwiGLU   (3행렬, 8/3x): {human(count_params(b))}")
print(f"{chr(10)}행렬이 3개지만 hidden 을 2/3 로 줄여서 파라미터를 비슷하게 맞춥니다.")
print("같은 예산으로 더 좋은 성능 -- 그래서 Llama, PaLM, Qwen 등이 채택했습니다.")

## 4. GQA — 추론 메모리를 줄이는 결정적 기법

이건 **학습보다 추론(서빙)에서 중요**합니다. 왜 그런지부터 봅시다.

### KV 캐시 문제

생성은 한 토큰씩 나아갑니다. 매 스텝 전체를 다시 계산하면 낭비죠.
그래서 앞선 토큰들의 **Key와 Value를 캐시**해 둡니다.

문제는 이 캐시가 **문맥 길이에 비례해 커진다**는 것.

In [ ]:
def kv_cache_gb(n_layer, n_head, head_dim, seq_len, batch=1, bytes_per=2):
    # K와 V 두 개, 2바이트(fp16) 기준
    return 2 * n_layer * n_head * head_dim * seq_len * batch * bytes_per / 1e9

# Llama-2 7B 급 설정
n_layer, n_head, head_dim = 32, 32, 128
print("Llama-2 7B 급 (32층, 32헤드, head_dim 128), fp16, 배치 1")
print(f"{'문맥 길이':>10} | {'KV 캐시 (MHA)':>16}")
print("-" * 32)
for T in (2048, 8192, 32768, 131072):
    print(f"{T:>10,} | {kv_cache_gb(n_layer, n_head, head_dim, T):>13.2f} GB")

print(f"{chr(10)}모델 가중치가 이미 약 14GB(fp16)입니다.")
print("문맥이 길어지면 캐시가 가중치보다 커집니다. 그리고 이건 사용자 1명당입니다.")

### 해결: Key와 Value를 헤드끼리 공유

- **MHA** (원래): Q, K, V 모두 헤드마다 따로 — 32헤드면 K/V도 32벌
- **MQA** (극단): K, V를 **딱 1벌**만 두고 모든 Q 헤드가 공유 — 캐시 32배 절감, 품질 저하
- **GQA** (절충): Q 헤드를 몇 개씩 묶어 **그룹마다 K/V 1벌** — 현대 표준

```
MHA (8 Q헤드, 8 KV헤드)   Q1 Q2 Q3 Q4 Q5 Q6 Q7 Q8
                          K1 K2 K3 K4 K5 K6 K7 K8

GQA (8 Q헤드, 2 KV그룹)   Q1 Q2 Q3 Q4 | Q5 Q6 Q7 Q8
                          └─── K1 ───┘ └─── K2 ───┘

MQA (8 Q헤드, 1 KV)       Q1 Q2 Q3 Q4 Q5 Q6 Q7 Q8
                          └────────── K1 ─────────┘
```

In [ ]:
print(f"{'방식':>16} | {'KV 헤드':>8} | {'32K 캐시':>12} | {'절감':>8}")
print("-" * 54)
base = kv_cache_gb(32, 32, 128, 32768)
for name, kvh in (("MHA (헤드 32)", 32), ("GQA (그룹 8)", 8), ("GQA (그룹 4)", 4), ("MQA (그룹 1)", 1)):
    g = kv_cache_gb(32, kvh, 128, 32768)
    print(f"{name:>16} | {kvh:>8} | {g:>9.2f} GB | {base/g:>7.0f}x")

print(f"{chr(10)}Llama-2 70B, Llama-3, Mistral 등이 GQA를 씁니다.")
print("추론 비용이 곧 서비스 원가라서, 이 절감이 그대로 돈입니다.")

In [ ]:
class GroupedQueryAttention(nn.Module):
    """GQA: Q는 n_head 개, K/V는 n_kv_head 개만 만들고 반복해서 씁니다."""
    def __init__(self, n_embd, n_head, n_kv_head):
        super().__init__()
        assert n_head % n_kv_head == 0
        self.n_head, self.n_kv_head = n_head, n_kv_head
        self.head_dim = n_embd // n_head
        self.n_rep = n_head // n_kv_head          # KV 하나를 몇 번 재사용할지

        self.wq = nn.Linear(n_embd, n_head * self.head_dim, bias=False)
        self.wk = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)   # 작음!
        self.wv = nn.Linear(n_embd, n_kv_head * self.head_dim, bias=False)   # 작음!
        self.wo = nn.Linear(n_embd, n_embd, bias=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.wq(x).view(B, T, self.n_head,    self.head_dim).transpose(1, 2)
        k = self.wk(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)
        v = self.wv(x).view(B, T, self.n_kv_head, self.head_dim).transpose(1, 2)

        # KV 헤드를 Q 헤드 수만큼 복제 (메모리에 캐시되는 건 복제 전의 작은 쪽)
        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)

        out = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.wo(out.transpose(1, 2).contiguous().view(B, T, C))

x = torch.randn(2, 10, 512)
for n_kv in (8, 4, 1):
    m = GroupedQueryAttention(512, n_head=8, n_kv_head=n_kv)
    print(f"n_kv_head={n_kv}: 출력 {tuple(m(x).shape)}, 파라미터 {human(count_params(m))}")

## 5. bias 제거

`nn.Linear(..., bias=False)` 가 계속 나왔습니다. 현대 모델은 대부분 편향을 뺍니다.

이유:
- 정규화 층이 이미 평균을 조정하고 있어서 **편향의 역할이 중복**
- 파라미터와 메모리 접근이 줄어듦
- 실험적으로 성능 저하가 없음

## 6. 다 합치면 — 현대식 블록

In [ ]:
class ModernBlock(nn.Module):
    """Pre-RMSNorm + GQA(RoPE) + SwiGLU. Llama 계열의 기본 형태."""
    def __init__(self, n_embd, n_head, n_kv_head):
        super().__init__()
        self.norm1 = RMSNorm(n_embd)
        self.attn  = GroupedQueryAttention(n_embd, n_head, n_kv_head)
        self.norm2 = RMSNorm(n_embd)
        self.ffn   = SwiGLU(n_embd)

    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

modern = ModernBlock(512, n_head=8, n_kv_head=2)
print(f"현대식 블록 출력: {tuple(modern(torch.randn(2, 10, 512)).shape)}")
print(f"파라미터: {human(count_params(modern))}")
print(f"{chr(10)}5-2의 두 줄 구조가 그대로 남아 있습니다:")
print("  x = x + attn(norm(x))")
print("  x = x + ffn(norm(x))")
print(f"{chr(10)}바뀐 건 각 부품의 '내용물'이지 뼈대가 아닙니다.")

---

## 정리

| 기법 | 무엇을 개선 | 체감되는 곳 |
|---|---|---|
| Pre-LN | 학습 안정성 | 발산 없이 깊게 학습 |
| RMSNorm | 속도 | 학습·추론 전반 |
| RoPE | 길이 확장성 | **128K 문맥이 가능한 이유** |
| SwiGLU | 표현력 | 같은 크기로 더 좋은 성능 |
| GQA | 추론 메모리 | **서빙 비용, 동시 사용자 수** |
| no-bias | 효율 | 미세하지만 누적 |

> **주목할 점**: 이 중 어느 것도 트랜스포머의 기본 아이디어를 바꾸지 않았습니다.
> 어텐션과 잔차의 뼈대는 2017년 그대로입니다.
> 8년간의 발전은 **같은 구조를 더 싸고 안정적으로 만든 것**이었습니다.

## 연습

1. `build_rope_cache` 의 `base` 를 10000 → 500000 으로 바꾸면 회전 주기가 어떻게 변하나요?
   (힌트: 긴 문맥 지원 모델들이 이 값을 키웁니다.)
2. `SwiGLU` 의 hidden 계산에서 `2/3` 를 빼면 파라미터가 얼마나 늘어나는지 확인하세요.
3. 자신이 쓰는 오픈 모델의 `config.json` 을 열어 `num_key_value_heads`, `rope_theta`,
   `hidden_act` 를 찾아보세요. 이 노트북의 용어가 그대로 나옵니다.

## 다음

→ `../Ch6/0601_MiniGPT.ipynb` : 지금까지의 부품으로 **작동하는 GPT** 만들기